# The evaluation set

This notebook reads the evaluation set: the tiles the selection kept, labelled by
the geology the ODE feature catalogue places over them, and drawn so that every
class holds as many tiles as every other.

## Setup

In [ ]:
"""Import the labels, the selection, and what reads them."""

from common.analysis.stats.artifacts import selection
from common.analysis.stats.dataset import aggregate
from common.analysis.stats.dataset import read as stats
from common.analysis.visualization.dataset.tables import landing, size
from common.building import paths
from common.building.metadata.read import read_observation_metadata
from evaluation.analysis import artifacts, configs, pair
from evaluation.analysis.visualization import tables
from evaluation.analysis.visualization.plots import classes, samples
from evaluation.building import configs as building
from evaluation.building import draw

WORKERS = 8

## The classes

A texture class holds every tile lying in the middle of one of its features, since
any patch of it shows what it is. An object class holds every tile an object of
the right size lies in whole, so that every one sits in its tile at a similar
scale. A tile two classes claim is left out, and so is a texture tile an object
reaches into.

### Every class and what it is read from

In [ ]:
"""Tabulate every class, the tiles it holds, and the ones drawn of them."""

labels = artifacts.read_labels()
tables.classes(labels, configs.load())

### Where the classes lie

Every drawn tile on the THEMIS mosaic, one colour per class.

In [ ]:
"""Map every drawn tile, one colour per class."""

classes.plot(selection.read_selection(), labels)

## What the instruments land on the drawn tiles

The same figures the training notebook reads off every kept tile, read here off
the drawn ones alone.

In [ ]:
"""Measure what the selection kept of every drawn tile."""

drawn = draw.drawn_selections(selection.read_selection(), labels)
read = aggregate.dataset_stats(stats.measure_every_tile(drawn, WORKERS))

In [ ]:
"""Tabulate what each instrument lands on a drawn tile and how far it reaches."""

landing.landed(read)

In [ ]:
"""Tabulate the drawn set the filter leaves."""

size.final(read)

## Two classes side by side

Read off the built evaluation dataset, which `./scripts/dh_dataset.sh evaluation`
brings down. Two classes are drawn at random every run, and one built tile of
each is set beside the other; name the two by hand to compare a chosen pair.

For the SHARAD pair, `ice_rich_plains` against `ice_poor_plains`, the difference
lies under the bright line of the surface echo in the radargram: a buried
reflector where the ice is, and nothing beneath the surface where it is not.

In [ ]:
"""Read the built set and draw two of its classes."""

root = paths.dataset_root(building.load_name())
records = read_observation_metadata(root)
built = [
    one for one in artifacts.read_labels(root) if one.tile in {r.tile for r in records}
]
compared = pair.random_pair(built)
compared

In [ ]:
"""Tabulate what the crops of the two classes hold."""

tables.crops(records, built, compared)

In [ ]:
"""Set one built tile of each class beside the other, instrument by instrument."""

samples.plot(root, records, built, pair.random_tiles(built, compared))